<a href="https://colab.research.google.com/github/sweeden-ttu/profile/blob/stable/notebooks/usd-llm/02_usd_qlora_unsloth_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · QLoRA prototype on Colab (Unsloth)

GPU runtime (L4 or A100). Phase 1 of the plan: a fast, cheap check that a small model can learn the repair task **before** spending cluster time on Megatron-Bridge.

**Status: scaffold, not yet run end-to-end.** Model ids, the Unsloth API and memory figures follow Unsloth's Gemma 4 guide as of October 2026; pin versions when you run it. Exit criterion: valid-parse rate and mean `usdfid` on the held-out families beat the zero-shot base model.

In [ ]:
!pip -q install unsloth usd-core tokenizers
!wget -q -O usdlab.py https://raw.githubusercontent.com/sweeden-ttu/profile/stable/notebooks/usd-llm/usdlab.py
import usdlab as U, json, torch
print(torch.cuda.get_device_name(0))

In [ ]:
# Recreate the data from notebook 01 (or upload training.jsonl / validation.jsonl)
rows = U.make_sft_pairs(700, seed=0)
test_fams = {'units', 'swap_label'}
train = [r for r in rows if r['family'] not in test_fams]; test = [r for r in rows if r['family'] in test_fams][:60]

In [ ]:
from unsloth import FastLanguageModel
MODEL = 'unsloth/gemma-4-E4B-it'   # E4B LoRA fits an L4; use a 26B/31B QLoRA only on A100 80GB
model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

In [ ]:
from datasets import Dataset
def fmt(r):
    msgs = [{'role': 'user', 'content': r['input']}, {'role': 'assistant', 'content': r['output']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}
ds = Dataset.from_list(train).map(fmt)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field='text', per_device_train_batch_size=1, gradient_accumulation_steps=8,
                   num_train_epochs=1, learning_rate=2e-4, logging_steps=10, output_dir='usd_lora',
                   bf16=torch.cuda.is_bf16_supported()))
# Train on completions only (the .usda answer), not on the prompt:
# from unsloth.chat_templates import train_on_responses_only  -> see Unsloth docs for the Gemma 4 markers
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)
import numpy as np
ref = test[0]['output']; scores, parsed = [], 0
for r in test:
    msgs = [{'role': 'user', 'content': r['input']}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = tokenizer.decode(model.generate(ids, max_new_tokens=3000, do_sample=False)[0][ids.shape[1]:], skip_special_tokens=True)
    res = U.usdfid(ref, out); parsed += res['L0_parse']; scores.append(res['score'])
print(f'valid parse {parsed}/{len(test)}  mean usdfid {np.mean(scores):.3f}')

In [ ]:
model.save_pretrained('usd_lora'); tokenizer.save_pretrained('usd_lora')   # adapter for the vLLM + ADK arena